# VNTS HLS — Final Evaluation & Statistical Comparison (Model A vs B vs P2)
Third notebook. Loads the three already-trained checkpoints (Model A, Model B, Model P2) and produces every number the report needs:

- Global Precision / Recall / mAP@0.5 / mAP@0.5:0.95, per model, on the held-out test set
- Tier-level mAP@0.5 (Frequent / Medium / Rare), per model
- Pairwise Wilcoxon signed-rank tests on paired per-class AP50 (A vs B, P2 vs B), per tier
- Per-QCVN-41-category mean AP50 delta (A vs B), plus the correlation check between raw instance count and delta
- Everything is evaluated on the test set **exactly once per model** — these are the same checkpoints and same test set used in notebooks 1 and 2's one-time evaluations, just re-run together here so the paired statistics (which need both models scored on identical per-class arrays) can be computed in one place.

**Update the paths in the next cell** to match wherever your checkpoints and the shared split actually live in this notebook's environment.

In [1]:
!pip install ultralytics scipy -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 42.1 MB/s eta 0:00:00


## Paths — update these to match your environment
`MODEL_PATHS` should point at the three `best.pt` files exported by notebooks 1 and 2. `SPLIT_DIR` should point at the shared split exported by notebook 1 (`exports/splits/`), specifically `test_data.yaml`, which encodes the exact 639-image held-out test set.

In [2]:
from pathlib import Path
import json
import numpy as np
import yaml
from ultralytics import YOLO
from scipy.stats import wilcoxon, pearsonr

# --- UPDATE THESE ---
DATASET = Path("/kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive")
LABELS  = DATASET / "labels"
CLASSES = DATASET / "classes_en.txt"

SPLIT_DIR = Path("/kaggle/input/datasets/banutheghostbanana/model-a-hls/exports/splits")   # from notebook 1
MODEL_PATHS = {
    "A":  Path("/kaggle/input/datasets/banutheghostbanana/model-a-hls/exports/model_A/model_A_best.pt"),
    "B":  Path("/kaggle/input/datasets/banutheghostbanana/model-b-and-p2/exports/model_B/model_B_best.pt"),
    "P2": Path("/kaggle/input/datasets/banutheghostbanana/model-b-and-p2/exports/model_P2/model_P2_best.pt"),
}
# ---------------------

WORK = Path("/kaggle/working")
WORK.mkdir(exist_ok=True, parents=True)

with open(CLASSES, encoding="utf-8") as f:
    class_names = [l.strip() for l in f if l.strip()]
n_classes = len(class_names)

test_yaml_path = SPLIT_DIR / "test_data.yaml"
with open(test_yaml_path, encoding="utf-8") as f:
    test_data = yaml.safe_load(f)

# Repoint "path" at this notebook's local copy of the split so the yaml resolves
LOCAL_SPLIT = WORK / "split"
LOCAL_SPLIT.mkdir(exist_ok=True, parents=True)
import shutil
shutil.copy(SPLIT_DIR / "test_paths.txt", LOCAL_SPLIT / "test_paths.txt")
test_data["path"] = str(LOCAL_SPLIT)
local_test_yaml = LOCAL_SPLIT / "test_data.yaml"
with open(local_test_yaml, "w", encoding="utf-8") as f:
    yaml.dump(test_data, f, allow_unicode=True, sort_keys=False)

print(f"Classes: {n_classes}")
print(f"Test yaml: {local_test_yaml}")
for name, p in MODEL_PATHS.items():
    print(f"Model {name}: {p}  (exists: {p.exists()})")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Classes: 52
Test yaml: /kaggle/working/split/test_data.yaml
Model A: /kaggle/input/datasets/banutheghostbanana/model-a-hls/exports/model_A/model_A_best.pt  (exists: True)
Model B: /kaggle/input/datasets/banutheghostbanana/model-b-and-p2/exports/model_B/model_B_best.pt  (exists: True)
Model P2: /kaggle/input/datasets/banutheghostbanana/model-b-and-p2/exports/model_P2/model_P2_best.pt  (exists: True)


## Frequency tiers and QCVN 41 category mapping
Recomputed deterministically (same logic as notebooks 1 and 2) rather than imported, since no randomness is involved.

In [3]:
with open(SPLIT_DIR / "train_paths.txt", encoding="utf-8") as f:
    all_paths = [l.strip() for l in f if l.strip()]
with open(SPLIT_DIR / "val_paths.txt", encoding="utf-8") as f:
    all_paths += [l.strip() for l in f if l.strip()]
with open(SPLIT_DIR / "test_paths.txt", encoding="utf-8") as f:
    all_paths += [l.strip() for l in f if l.strip()]

def count_instances(paths):
    counts = np.zeros(n_classes, dtype=int)
    for p in paths:
        label_path = LABELS / (Path(p).stem + ".txt")
        if label_path.exists():
            with open(label_path, encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        counts[int(line.split()[0])] += 1
    return counts

all_counts = count_instances(all_paths)

FREQ_CLASS_IDS = np.where(all_counts >= 100)[0]
MED_CLASS_IDS  = np.where((all_counts >= 30) & (all_counts < 100))[0]
RARE_CLASS_IDS = np.where(all_counts < 30)[0]

TIER_IDS = {"global": np.arange(n_classes), "freq": FREQ_CLASS_IDS,
            "med": MED_CLASS_IDS, "rare": RARE_CLASS_IDS}

print(f"Frequent: {len(FREQ_CLASS_IDS)} | Medium: {len(MED_CLASS_IDS)} | Rare: {len(RARE_CLASS_IDS)}")

# QCVN 41 Level-1 category mapping (0=Prohibitory, 1=Warning, 2=Mandatory,
# 3=Information, 4=Supplementary) -- identical to notebook 1.
CATEGORY_MAPPING = {
    2:0, 7:0, 10:0, 14:0, 16:0, 17:0, 18:0, 19:0, 20:0,
    23:0, 24:0, 29:0, 32:0, 34:0, 35:0, 36:0, 38:0, 39:0,
    40:0, 41:0, 46:0, 47:0, 49:0,
    0:1, 1:1, 4:1, 5:1, 6:1, 13:1, 15:1, 21:1, 22:1,
    26:1, 27:1, 33:1, 37:1, 43:1, 44:1, 48:1, 50:1, 51:1,
    3:2, 9:2, 12:2, 42:2,
    8:3, 11:3, 31:3, 45:3,
    25:4, 28:4, 30:4,
}
CATEGORY_NAMES = {0: "Prohibitory", 1: "Warning", 2: "Mandatory",
                   3: "Information", 4: "Supplementary"}


Frequent: 24 | Medium: 15 | Rare: 13


## Evaluate all three models on the test set (once each)
Extracts the full-length, class-ID-aligned per-class AP50 array for each model (same `ap_class_index` fix from notebooks 1/2), plus global Precision/Recall/mAP50/mAP50-95.

In [4]:
def evaluate_model(weights_path):
    model = YOLO(str(weights_path))
    metrics = model.val(data=str(local_test_yaml), split="val")

    # Global scalar metrics (Ultralytics macro-averages precision/recall over classes)
    global_metrics = {
        "precision": float(metrics.box.mp),
        "recall":    float(metrics.box.mr),
        "map50":     float(metrics.box.map50),
        "map50_95":  float(metrics.box.map),
    }

    # Per-class AP50, aligned back to true class IDs (classes absent from this
    # val split's ground truth are NaN, not silently dropped/misaligned).
    full_ap50 = np.full(n_classes, np.nan)
    ap50_present = metrics.box.all_ap[:, 0]
    class_idx = np.asarray(metrics.box.ap_class_index)
    full_ap50[class_idx] = ap50_present

    tier_scores = {
        tier: float(np.nanmean(full_ap50[ids])) for tier, ids in TIER_IDS.items()
    }

    return {
        "global_metrics": global_metrics,
        "tier_scores": tier_scores,
        "per_class_ap50": full_ap50,
    }

results = {}
for name, path in MODEL_PATHS.items():
    print(f"\n=== Evaluating Model {name} ===")
    results[name] = evaluate_model(path)
    print(json.dumps(results[name]["global_metrics"], indent=2))
    print(json.dumps(results[name]["tier_scores"], indent=2))



=== Evaluating Model A ===
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,384,976 parameters, 0 gradients, 5.2 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 5.8±2.0 ms, read: 27.9±11.0 MB/s, size: 239.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive/labels... 639 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 639/639 221.2it/s 2.9s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 40/40 6.6it/s 6.1s0.1ss
                   all        639       1645      0.829      0.869      0.913      0.669
   Pedestrian Crossing         61         61    

## Pairwise Wilcoxon signed-rank tests (A vs B, P2 vs B)
Paired by class ID, per tier. Classes missing (NaN) in either model's array for that pair are dropped from that specific test — `scipy.stats.wilcoxon` can't handle NaNs, and a class absent from ground truth for one model isn't comparable anyway.

In [5]:
def paired_wilcoxon(ap_x, ap_y, class_ids):
    x = ap_x[class_ids]
    y = ap_y[class_ids]
    valid = ~(np.isnan(x) | np.isnan(y))
    x, y = x[valid], y[valid]
    n = len(x)
    if n < 1 or np.allclose(x, y):
        return {"n": n, "p_value": None}
    try:
        stat, p = wilcoxon(x, y)
        return {"n": n, "p_value": float(p)}
    except ValueError:
        return {"n": n, "p_value": None}

comparisons = {}
for pair_name, (m1, m2) in [("A_vs_B", ("A", "B")), ("P2_vs_B", ("P2", "B"))]:
    comparisons[pair_name] = {
        tier: paired_wilcoxon(
            results[m1]["per_class_ap50"], results[m2]["per_class_ap50"], ids
        )
        for tier, ids in TIER_IDS.items()
    }
    print(f"\n=== {pair_name.replace('_', ' ')} ===")
    for tier, res in comparisons[pair_name].items():
        p_str = f"{res['p_value']:.4f}" if res["p_value"] is not None else "N/A"
        print(f"  {tier:8s} n={res['n']:3d}  p={p_str}")



=== A vs B ===
  global   n= 52  p=0.5539
  freq     n= 24  p=0.8774
  med      n= 15  p=0.8139
  rare     n= 13  p=0.2439

=== P2 vs B ===
  global   n= 52  p=0.0000
  freq     n= 24  p=0.0000
  med      n= 15  p=0.0131
  rare     n= 13  p=0.0645


## Mechanism analysis: per-category delta (A vs B) and instance-count correlation
Mirrors the original report's Section 5.5 — checks whether Model A's improvement over Model B tracks QCVN 41 category size rather than raw class rarity.

In [6]:
delta_a_minus_b = results["A"]["per_class_ap50"] - results["B"]["per_class_ap50"]

category_sizes = {cat: sum(1 for v in CATEGORY_MAPPING.values() if v == cat) for cat in CATEGORY_NAMES}
category_deltas = {}
for cat_id, cat_name in CATEGORY_NAMES.items():
    class_ids = [c for c, cat in CATEGORY_MAPPING.items() if cat == cat_id]
    deltas = delta_a_minus_b[class_ids]
    valid = deltas[~np.isnan(deltas)]
    category_deltas[cat_name] = {
        "n_classes": category_sizes[cat_id],
        "mean_delta": float(np.mean(valid)) if len(valid) else None,
    }

print("=== Mean AP50 delta (A - B) by QCVN 41 category ===")
for cat_name, d in sorted(category_deltas.items(), key=lambda kv: kv[1]["n_classes"]):
    md = f"{d['mean_delta']:+.4f}" if d["mean_delta"] is not None else "N/A"
    print(f"  {cat_name:14s} ({d['n_classes']:2d} classes): {md}")

# Correlation: raw training-instance count vs AP50 delta
valid_classes = ~np.isnan(delta_a_minus_b)
r, p_corr = pearsonr(all_counts[valid_classes], delta_a_minus_b[valid_classes])
print(f"\nPearson r (instance count vs delta): {r:.4f}  (p={p_corr:.4f})")


=== Mean AP50 delta (A - B) by QCVN 41 category ===
  Supplementary  ( 3 classes): -0.1527
  Mandatory      ( 4 classes): +0.0281
  Information    ( 4 classes): +0.0344
  Warning        (18 classes): -0.0093
  Prohibitory    (23 classes): -0.0120

Pearson r (instance count vs delta): 0.0737  (p=0.6037)


## Export everything for the report
Writes a single JSON with every number above, plus a plain-text summary you can paste directly into a message to fill in the report's tables.

In [7]:
export = {
    "global_metrics": {m: results[m]["global_metrics"] for m in results},
    "tier_scores":     {m: results[m]["tier_scores"] for m in results},
    "per_class_ap50":  {m: results[m]["per_class_ap50"].tolist() for m in results},
    "wilcoxon":        comparisons,
    "category_deltas": category_deltas,
    "instance_count_correlation": {"r": float(r), "p_value": float(p_corr)},
}

EXPORT_PATH = WORK / "final_evaluation_results.json"
with open(EXPORT_PATH, "w") as f:
    json.dump(export, f, indent=2)

print(f"Exported to: {EXPORT_PATH}\n")
print("=" * 70)
print("SUMMARY (paste this back into chat)")
print("=" * 70)
for m in ["A", "B", "P2"]:
    g = results[m]["global_metrics"]
    t = results[m]["tier_scores"]
    print(f"\nModel {m}:")
    print(f"  Precision={g['precision']:.4f}  Recall={g['recall']:.4f}  "
          f"mAP50={g['map50']:.4f}  mAP50-95={g['map50_95']:.4f}")
    print(f"  Tier mAP50 -- Global={t['global']:.4f}  Freq={t['freq']:.4f}  "
          f"Med={t['med']:.4f}  Rare={t['rare']:.4f}")

print("\nWilcoxon (A vs B):")
for tier, res in comparisons["A_vs_B"].items():
    print(f"  {tier:8s} n={res['n']:3d}  p={res['p_value']}")

print("\nWilcoxon (P2 vs B):")
for tier, res in comparisons["P2_vs_B"].items():
    print(f"  {tier:8s} n={res['n']:3d}  p={res['p_value']}")

print("\nCategory deltas (A - B):")
for cat_name, d in category_deltas.items():
    md_val = d["mean_delta"]
    print(f"  {cat_name:14s}: {md_val}")

print(f"\nInstance-count correlation: r={r:.4f}, p={p_corr:.4f}")


Exported to: /kaggle/working/final_evaluation_results.json

SUMMARY (paste this back into chat)

Model A:
  Precision=0.8295  Recall=0.8689  mAP50=0.9131  mAP50-95=0.6690
  Tier mAP50 -- Global=0.9131  Freq=0.9556  Med=0.9387  Rare=0.8053

Model B:
  Precision=0.8337  Recall=0.8796  mAP50=0.9257  mAP50-95=0.6945
  Tier mAP50 -- Global=0.9257  Freq=0.9536  Med=0.9336  Rare=0.8649

Model P2:
  Precision=0.8455  Recall=0.6708  mAP50=0.8335  mAP50-95=0.6173
  Tier mAP50 -- Global=0.8335  Freq=0.8839  Med=0.8523  Rare=0.7188

Wilcoxon (A vs B):
  global   n= 52  p=0.5539391908305951
  freq     n= 24  p=0.8774070739746094
  med      n= 15  p=0.813875058383029
  rare     n= 13  p=0.243896484375

Wilcoxon (P2 vs B):
  global   n= 52  p=3.3857754756564326e-07
  freq     n= 24  p=3.5762786865234375e-07
  med      n= 15  p=0.013103614110038467
  rare     n= 13  p=0.064453125

Category deltas (A - B):
  Prohibitory   : -0.012004749369955277
  Mandatory     : 0.028116456071539064
  Information   : 